# Tutorial 11A: Introduction to Stochastic Frontier Analysis

**Module**: Stochastic Frontier Analysis | **Difficulty**: Beginner | **Duration**: 2-3 hours

**Prerequisites**: Basic econometrics (OLS, MLE), production function concepts, Python basics

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Understand the fundamental concepts of Stochastic Frontier Analysis (SFA)
2. Explain the difference between OLS and SFA for efficiency measurement
3. Interpret the composed error structure ($v - u$)
4. Estimate basic cross-section SFA models using PanelBox
5. Calculate technical efficiency scores using JLMS and BC estimators
6. Visualize frontiers and efficiency distributions
7. Compare different distributional assumptions for inefficiency
8. Conduct tests for the presence of inefficiency

## Table of Contents

1. [Economic Motivation](#1-economic-motivation)
2. [Theoretical Framework](#2-theoretical-framework)
3. [Setup and Data Loading](#3-setup-and-data-loading)
4. [OLS Baseline](#4-ols-baseline)
5. [Basic SFA Estimation](#5-basic-sfa-estimation)
6. [Testing for Inefficiency](#6-testing-for-inefficiency)
7. [Efficiency Estimation](#7-efficiency-estimation)
8. [Efficiency Visualization](#8-efficiency-visualization)
9. [Frontier Visualization](#9-frontier-visualization)
10. [Comparing Distributional Assumptions](#10-comparing-distributional-assumptions)
11. [Exercises](#11-exercises)
12. [Summary](#12-summary)
13. [References](#13-references)

## 1. Economic Motivation

<a id="1-economic-motivation"></a>

### Why Do We Need Stochastic Frontier Analysis?

Consider two hospitals operating in the same region:

- **Hospital A** treats 5,000 cases per year with 50 doctors, 100 nurses, and 200 beds
- **Hospital B** treats 3,500 cases per year with 50 doctors, 100 nurses, and 200 beds

Hospital B produces 30% less output with the **same inputs**. But why?

There are two possible explanations:
1. **Bad luck (noise)**: Hospital B may face a flu outbreak, equipment failures, or unusually complex cases — random factors beyond its control.
2. **Inefficiency**: Hospital B may have poor management, waste resources, or use outdated procedures — factors that could be improved.

**Ordinary Least Squares (OLS)** treats all deviations from the average as random noise. It estimates the *average* production function, not the *best-practice* frontier. This means OLS cannot distinguish between bad luck and genuine inefficiency.

**Stochastic Frontier Analysis (SFA)** solves this by decomposing the error term into two components:

```
Output |    * Frontier: y = f(x) + v
       |  *    *
       |     *    <- v (symmetric noise: luck, measurement error)
       |  *       <- u (one-sided inefficiency: always reduces output)
       |    *
       |  *
       |_____________________
              Input
```

The key insight of SFA is the **composed error** $\varepsilon = v - u$, where:
- $v$ is **symmetric noise** (can be positive or negative) — captures random shocks, measurement error, luck
- $u \geq 0$ is **one-sided inefficiency** (always non-negative for production frontiers) — captures technical inefficiency

This allows us to estimate:
- The **production frontier** (best-practice technology)
- **Technical efficiency** scores for each observation
- The relative importance of noise vs. inefficiency

### Why Not Just Use Data Envelopment Analysis (DEA)?

DEA is a popular non-parametric alternative, but it attributes ALL deviations from the frontier to inefficiency. In noisy environments (like healthcare), this can severely overestimate inefficiency. SFA's statistical framework explicitly accounts for noise, making it more appropriate when random shocks are present.

## 2. Theoretical Framework

<a id="2-theoretical-framework"></a>

### 2.1 The Stochastic Production Frontier

The **deterministic** production frontier is:

$$y_i = f(x_i; \beta)$$

where $y_i$ is output, $x_i$ is a vector of inputs, and $\beta$ are technology parameters. In practice, output is subject to random shocks, so we add a noise term:

$$y_i = f(x_i; \beta) \cdot \exp(v_i)$$

This is the **stochastic** frontier. But firms may also be **inefficient**, producing below the frontier:

$$y_i = f(x_i; \beta) \cdot \exp(v_i) \cdot \exp(-u_i)$$

where $u_i \geq 0$ represents technical inefficiency. Taking logarithms of a Cobb-Douglas specification:

$$\ln y_i = \beta_0 + \sum_k \beta_k \ln x_{ki} + v_i - u_i$$

or equivalently:

$$\ln y_i = \mathbf{x}_i' \boldsymbol{\beta} + \varepsilon_i, \quad \text{where } \varepsilon_i = v_i - u_i$$

### 2.2 The Composed Error Structure

The composed error $\varepsilon_i = v_i - u_i$ has two components:

| Component | Symbol | Distribution | Sign | Interpretation |
|-----------|--------|-------------|------|----------------|
| Noise | $v_i$ | $N(0, \sigma_v^2)$ | $\pm$ | Random shocks, luck, measurement error |
| Inefficiency | $u_i$ | $u_i \geq 0$ | $\geq 0$ | Technical inefficiency (always reduces output) |

**Key implication**: Since $u_i \geq 0$, the composed error $\varepsilon_i$ is **negatively skewed** for production frontiers. This asymmetry is what allows us to separate noise from inefficiency.

### 2.3 Technical Efficiency

Technical efficiency for firm $i$ is defined as:

$$TE_i = \frac{y_i}{f(x_i; \beta) \cdot \exp(v_i)} = \exp(-u_i) \in (0, 1]$$

- $TE_i = 1$: Firm operates **on** the frontier (fully efficient)
- $TE_i < 1$: Firm operates **below** the frontier (inefficient)
- $TE_i = 0.85$: Firm produces 85% of what the frontier predicts, given its inputs and noise realization

### 2.4 Distributional Assumptions for $u_i$

Different distributional assumptions for $u_i$ lead to different models:

| Distribution | Density | Parameters | Properties |
|-------------|---------|-----------|------------|
| **Half-normal** | $f(u) = \frac{2}{\sqrt{2\pi}\sigma_u} \exp\left(-\frac{u^2}{2\sigma_u^2}\right)$ | $\sigma_u > 0$ | Mode at zero; most firms near-efficient |
| **Exponential** | $f(u) = \frac{1}{\sigma_u} \exp\left(-\frac{u}{\sigma_u}\right)$ | $\sigma_u > 0$ | Mode at zero; thinner tail than half-normal |
| **Truncated normal** | $f(u) = \frac{\phi\left(\frac{u - \mu}{\sigma_u}\right)}{\sigma_u \left[1 - \Phi\left(-\frac{\mu}{\sigma_u}\right)\right]}$ | $\mu, \sigma_u$ | Mode can be away from zero; more flexible |
| **Gamma** | $f(u) = \frac{u^{P-1} \exp(-u/\sigma_u)}{\sigma_u^P \Gamma(P)}$ | $P, \sigma_u$ | Very flexible; nests exponential ($P=1$) |

The **half-normal** distribution is the most common starting point. It implies most firms are relatively efficient (mode at zero).

### 2.5 Efficiency Estimators: JLMS and BC

Once the model is estimated, we need to recover firm-level efficiency. Two main approaches exist:

**JLMS Estimator** (Jondrow, Lovell, Materov, Schmidt, 1982):

$$E[u_i | \varepsilon_i] = \frac{\sigma_u \sigma_v}{\sigma} \left[ \frac{\phi(\varepsilon_i \lambda / \sigma)}{\Phi(-\varepsilon_i \lambda / \sigma)} - \frac{\varepsilon_i \lambda}{\sigma} \right]$$

where $\lambda = \sigma_u / \sigma_v$ and $\sigma^2 = \sigma_u^2 + \sigma_v^2$.

**BC Estimator** (Battese and Coelli, 1988):

$$TE_i^{BC} = E[\exp(-u_i) | \varepsilon_i] = \frac{\Phi\left(-\eta_i \sigma_* + \sigma_*\right)}{\Phi\left(-\eta_i \sigma_*\right)} \cdot \exp\left(-\eta_i \mu_{*i} + \frac{1}{2}\sigma_*^2\right)$$

where $\mu_{*i}$ and $\sigma_*^2$ are the conditional mean and variance of $u_i | \varepsilon_i$.

**Why prefer BC over JLMS?**
- JLMS estimates $E[u_i | \varepsilon_i]$, then exponentiates: $\widehat{TE}_i^{JLMS} = \exp(-E[u_i | \varepsilon_i])$
- BC directly estimates $E[\exp(-u_i) | \varepsilon_i]$
- By Jensen's inequality, $\exp(-E[u]) \neq E[\exp(-u)]$, and the BC estimator is **less biased**

### 2.6 Variance Decomposition

Two key ratios characterize the relative importance of inefficiency:

$$\lambda = \frac{\sigma_u}{\sigma_v}$$

- $\lambda > 1$: Inefficiency dominates noise
- $\lambda < 1$: Noise dominates inefficiency

$$\gamma = \frac{\sigma_u^2}{\sigma_u^2 + \sigma_v^2} \in [0, 1]$$

- $\gamma \to 1$: Most variance is due to inefficiency (frontier model is appropriate)
- $\gamma \to 0$: Most variance is due to noise (OLS would suffice)

In [ ]:
# ============================================================
# Section 3: Setup and Data Loading
# ============================================================

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

import statsmodels.api as sm
from scipy import stats

from panelbox.frontier import StochasticFrontier
from panelbox.frontier.tests import inefficiency_presence_test

np.random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "01_intro"
TABLES_DIR_LATEX = BASE_DIR / "outputs" / "tables" / "latex"
TABLES_DIR_HTML = BASE_DIR / "outputs" / "tables" / "html"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR_LATEX.mkdir(parents=True, exist_ok=True)
TABLES_DIR_HTML.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
# Load the hospital dataset
data = pd.read_csv(DATA_DIR / "hospital_data.csv")

print(f"Dataset shape: {data.shape}")
print(f"Variables: {data.columns.tolist()}")
print("\nFirst 5 observations:")
display(data.head())

print("\nSummary Statistics:")
display(data.describe())

print(f"\nMissing values: {data.isnull().sum().sum()}")

### Understanding the Variables

Our hospital dataset contains the following variables:

| Variable | Description | Type |
|----------|------------|------|
| `hospital_id` | Unique hospital identifier | Identifier |
| `log_cases` | Log of cases treated (output) | Continuous |
| `log_doctors` | Log of number of doctors (input) | Continuous |
| `log_nurses` | Log of number of nurses (input) | Continuous |
| `log_beds` | Log of number of beds (input) | Continuous |
| `teaching` | Teaching hospital indicator (1 = yes) | Binary |
| `urban` | Urban location indicator (1 = yes) | Binary |
| `region` | Geographic region | Categorical |

The log-log specification implies a **Cobb-Douglas production function**:

$$\text{Cases} = A \cdot \text{Doctors}^{\beta_1} \cdot \text{Nurses}^{\beta_2} \cdot \text{Beds}^{\beta_3} \cdot \exp(v - u)$$

where $\beta_k$ are output elasticities with respect to each input.

In [ ]:
# Exploratory visualizations: distributions of key variables
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

variables = ["log_cases", "log_doctors", "log_nurses", "log_beds"]
titles = [
    "Log Cases Treated (Output)",
    "Log Doctors (Input)",
    "Log Nurses (Input)",
    "Log Beds (Input)",
]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

for ax, var, title, color in zip(axes.flat, variables, titles, colors):
    ax.hist(
        data[var], bins=25, density=True, alpha=0.7, color=color, edgecolor="black", linewidth=0.5
    )
    # Add KDE
    kde = stats.gaussian_kde(data[var])
    x_range = np.linspace(data[var].min(), data[var].max(), 200)
    ax.plot(x_range, kde(x_range), color="black", linewidth=2)
    ax.axvline(
        data[var].mean(),
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=f"Mean: {data[var].mean():.2f}",
    )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel(var)
    ax.set_ylabel("Density")
    ax.legend(fontsize=10)

fig.suptitle("Distribution of Key Variables", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "variable_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap
numeric_vars = ["log_cases", "log_doctors", "log_nurses", "log_beds", "teaching", "urban"]
corr_matrix = data[numeric_vars].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=1,
    ax=ax,
    vmin=-1,
    vmax=1,
    cbar_kws={"label": "Correlation"},
)
ax.set_title("Correlation Matrix of Hospital Variables", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nKey observations:")
print(
    f"  - Correlation between log_cases and log_doctors: {corr_matrix.loc['log_cases', 'log_doctors']:.3f}"
)
print(
    f"  - Correlation between log_cases and log_nurses:  {corr_matrix.loc['log_cases', 'log_nurses']:.3f}"
)
print(
    f"  - Correlation between log_cases and log_beds:    {corr_matrix.loc['log_cases', 'log_beds']:.3f}"
)

## 4. OLS Baseline

<a id="4-ols-baseline"></a>

Before estimating the stochastic frontier, we establish an OLS baseline. This serves two purposes:

1. **Comparison**: We can compare SFA coefficients with OLS to see how accounting for inefficiency changes the estimated technology parameters.
2. **Diagnostic**: The **skewness of OLS residuals** provides a first indication of whether inefficiency is present. For production frontiers, we expect *negative* skewness (since $u \geq 0$ pulls residuals to the left).

If OLS residuals are symmetric, there may be no evidence of inefficiency, and the SFA model may not be appropriate.

In [ ]:
# OLS estimation as baseline
X_ols = sm.add_constant(data[["log_doctors", "log_nurses", "log_beds", "teaching", "urban"]])
ols_model = sm.OLS(data["log_cases"], X_ols).fit()

print("=" * 70)
print("OLS ESTIMATION RESULTS")
print("=" * 70)
print(ols_model.summary())

# Examine residual skewness
residuals = ols_model.resid
skewness = stats.skew(residuals)
kurtosis = stats.kurtosis(residuals)

print(f"\n{'=' * 50}")
print("Residual Diagnostics:")
print(f"{'=' * 50}")
print(f"  Skewness:  {skewness:.4f}  (expect < 0 for production frontier)")
print(f"  Kurtosis:  {kurtosis:.4f}  (excess, normal = 0)")
print(
    f"  Jarque-Bera test: stat = {stats.jarque_bera(residuals)[0]:.4f}, p = {stats.jarque_bera(residuals)[1]:.4f}"
)

if skewness < 0:
    print("\n  >> Negative skewness detected! This is consistent with")
    print("     the presence of one-sided inefficiency (u >= 0).")
else:
    print("\n  >> WARNING: Positive skewness detected. This is unexpected")
    print("     for a production frontier. Proceed with caution.")

# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of residuals
axes[0].hist(
    residuals,
    bins=30,
    density=True,
    alpha=0.7,
    color="steelblue",
    edgecolor="black",
    linewidth=0.5,
    label="OLS Residuals",
)
# Normal density for comparison
x_norm = np.linspace(residuals.min(), residuals.max(), 200)
axes[0].plot(
    x_norm,
    stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
    "r-",
    linewidth=2,
    label="Normal density",
)
# KDE
kde = stats.gaussian_kde(residuals)
axes[0].plot(x_norm, kde(x_norm), "g--", linewidth=2, label="KDE")
axes[0].axvline(0, color="black", linestyle=":", alpha=0.5)
axes[0].set_title("Distribution of OLS Residuals", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Residual")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].annotate(
    f"Skewness = {skewness:.3f}",
    xy=(0.05, 0.95),
    xycoords="axes fraction",
    fontsize=11,
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
    verticalalignment="top",
)

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot of OLS Residuals", fontsize=13, fontweight="bold")
axes[1].get_lines()[0].set_markerfacecolor("steelblue")
axes[1].get_lines()[0].set_markeredgecolor("steelblue")
axes[1].get_lines()[0].set_markersize(4)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "ols_residuals.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation of OLS Results

The OLS estimation provides a baseline for comparison:

- **Output elasticities** ($\hat{\beta}_k$): These tell us the percentage change in output for a 1% change in each input, *on average*. However, OLS estimates the *average* production function, not the frontier.
- **Residual skewness**: Negative skewness is consistent with the SFA model, as the one-sided inefficiency term ($u \geq 0$) pulls residuals leftward. This provides preliminary evidence that inefficiency may be present.
- **R-squared**: Measures the proportion of variance explained by inputs, but does not account for inefficiency.

**Key limitation**: OLS treats the entire residual as noise. It cannot tell us whether Hospital B (from our earlier example) is inefficient or just unlucky. This motivates the SFA approach.

## 5. Basic SFA Estimation

<a id="5-basic-sfa-estimation"></a>

Now we estimate the stochastic production frontier using Maximum Likelihood Estimation (MLE). We start with the **half-normal** distribution for the inefficiency term, which is the most common specification.

The model is:

$$\ln(\text{cases}_i) = \beta_0 + \beta_1 \ln(\text{doctors}_i) + \beta_2 \ln(\text{nurses}_i) + \beta_3 \ln(\text{beds}_i) + \beta_4 \text{teaching}_i + \beta_5 \text{urban}_i + v_i - u_i$$

where $v_i \sim N(0, \sigma_v^2)$ and $u_i \sim N^+(0, \sigma_u^2)$ (half-normal).

In [ ]:
# Estimate the Stochastic Production Frontier
sfa_model = StochasticFrontier(
    data=data,
    depvar="log_cases",
    exog=["log_doctors", "log_nurses", "log_beds", "teaching", "urban"],
    frontier="production",
    dist="half_normal",
)

result = sfa_model.fit(method="mle", optimizer="L-BFGS-B")

print("=" * 70)
print("STOCHASTIC FRONTIER ANALYSIS - HALF-NORMAL MODEL")
print("=" * 70)
print(result.summary())

In [ ]:
# Variance decomposition
print("=" * 60)
print("VARIANCE DECOMPOSITION")
print("=" * 60)

print(f"\n  sigma_v (noise std. dev.):       {result.sigma_v:.4f}")
print(f"  sigma_u (inefficiency std. dev.): {result.sigma_u:.4f}")
print(f"  sigma   (total std. dev.):        {result.sigma:.4f}")
print(f"  lambda  (sigma_u / sigma_v):      {result.lambda_param:.4f}")
print(f"  gamma   (sigma_u^2 / sigma^2):    {result.gamma:.4f}")

# Detailed variance decomposition
vd = result.variance_decomposition()
print(f"\n{'=' * 60}")
print("Detailed Variance Decomposition:")
print(f"{'=' * 60}")
print(f"  sigma_sq_v (noise variance):        {vd['sigma_sq_v']:.6f}")
print(f"  sigma_sq_u (inefficiency variance):  {vd['sigma_sq_u']:.6f}")
print(f"  sigma_sq   (total variance):         {vd['sigma_sq']:.6f}")
print(f"\n  gamma = {vd['gamma']:.4f}")
if "gamma_ci" in vd and vd["gamma_ci"] is not None:
    print(f"  gamma 95% CI: [{vd['gamma_ci'][0]:.4f}, {vd['gamma_ci'][1]:.4f}]")
print(f"\n  lambda = {vd['lambda_param']:.4f}")
if "lambda_ci" in vd and vd["lambda_ci"] is not None:
    print(f"  lambda 95% CI: [{vd['lambda_ci'][0]:.4f}, {vd['lambda_ci'][1]:.4f}]")
print(f"\n  Interpretation: {vd['interpretation']}")

# Model fit statistics
print(f"\n{'=' * 60}")
print("Model Fit Statistics:")
print(f"{'=' * 60}")
print(f"  Log-likelihood: {result.loglik:.4f}")
print(f"  AIC:            {result.aic:.4f}")
print(f"  BIC:            {result.bic:.4f}")
print(f"  N observations: {result.nobs}")
print(f"  N parameters:   {result.nparams}")
print(f"  Converged:      {result.converged}")

### Interpretation of SFA Results

Let's interpret the key results:

**Technology Parameters (Frontier Coefficients)**:
- The coefficients on the log-inputs are **output elasticities**. They tell us the percentage change in output when an input increases by 1%, holding other inputs constant.
- Compare these with the OLS estimates. SFA coefficients should be *different* because SFA estimates the **frontier** (best practice), not the average.
- The sum of input elasticities gives **returns to scale** (RTS): if $\sum \beta_k > 1$, we have increasing returns; if $= 1$, constant returns; if $< 1$, decreasing returns.

**Variance Parameters**:
- **$\sigma_u$** (sigma_u): Standard deviation of inefficiency. Larger values mean more inefficiency in the sample.
- **$\sigma_v$** (sigma_v): Standard deviation of noise. Represents the magnitude of random shocks.
- **$\lambda = \sigma_u / \sigma_v$**: If $\lambda > 1$, inefficiency dominates noise. This suggests the frontier model is appropriate.
- **$\gamma = \sigma_u^2 / (\sigma_u^2 + \sigma_v^2)$**: The proportion of total variance attributable to inefficiency. Values close to 1 indicate most variation comes from inefficiency, not noise.

**Comparison with OLS**:
- SFA frontier coefficients tend to be *larger* than OLS because OLS estimates the average, while SFA estimates the frontier (upper boundary).
- The SFA intercept is typically *higher* than OLS because it represents the frontier level, not the average level.

## 6. Testing for Inefficiency

<a id="6-testing-for-inefficiency"></a>

Before interpreting efficiency scores, we should formally test whether inefficiency is statistically significant. The null hypothesis is:

$$H_0: \sigma_u^2 = 0 \quad \text{(no inefficiency; OLS is adequate)}$$
$$H_1: \sigma_u^2 > 0 \quad \text{(inefficiency is present; SFA is needed)}$$

This is a **one-sided test** because $\sigma_u^2$ is on the boundary of the parameter space under $H_0$. The standard likelihood ratio test statistic follows a **mixed chi-squared** distribution:

$$LR = -2[\ln L(\text{OLS}) - \ln L(\text{SFA})] \sim \frac{1}{2}\chi^2(0) + \frac{1}{2}\chi^2(1)$$

This mixed distribution arises because $\sigma_u^2 = 0$ is at the boundary of the parameter space (it cannot be negative).

In [ ]:
# Test for the presence of inefficiency
test_result = inefficiency_presence_test(
    loglik_sfa=result.loglik,
    loglik_ols=ols_model.llf,
    residuals_ols=ols_model.resid.values,
    frontier_type="production",
    distribution="half_normal",
)

print("=" * 60)
print("TEST FOR THE PRESENCE OF INEFFICIENCY")
print("=" * 60)
print(f"\n  Likelihood Ratio Statistic: {test_result['lr_statistic']:.4f}")
print(f"  Degrees of Freedom:         {test_result['df']}")
print(f"  P-value:                     {test_result['pvalue']:.6f}")

print("\n  Critical Values:")
for level, cv in test_result["critical_values"].items():
    print(f"    {level}: {cv:.4f}")

print(f"\n  OLS Residual Skewness: {test_result['skewness']:.4f}")
if test_result.get("skewness_warning"):
    print(f"  WARNING: {test_result['skewness_warning']}")

print(f"\n  Conclusion: {test_result['conclusion']}")

### Understanding the Inefficiency Test

**The Mixed Chi-Squared Distribution**

The test statistic $LR = -2[\ln L_{OLS} - \ln L_{SFA}]$ does **not** follow a standard $\chi^2$ distribution because the null hypothesis places $\sigma_u^2$ at the **boundary** of the parameter space (zero). Under the null:

$$LR \sim \frac{1}{2}\chi^2(0) + \frac{1}{2}\chi^2(1)$$

This means:
- With probability 0.5, $LR = 0$ (the mass point at zero from $\chi^2(0)$)
- With probability 0.5, $LR$ follows a $\chi^2(1)$ distribution

The critical values for this mixed distribution are **smaller** than standard $\chi^2(1)$ critical values. For example, at 5% significance, the critical value is approximately 2.706 (compared to 3.841 for $\chi^2(1)$).

**Residual Skewness Check**

The test also reports the skewness of OLS residuals. For a production frontier, we expect **negative skewness** (because $u \geq 0$ shifts the distribution left). If skewness is positive ("wrong" sign), this may indicate:
- The sample is too small to detect inefficiency
- The functional form is misspecified
- A cost frontier (positive skewness) may be more appropriate

**Interpreting the Result**

If the test rejects $H_0$ (p-value < 0.05), we conclude that inefficiency is statistically significant and the SFA model is preferred over OLS. If we fail to reject, OLS may be adequate for this data.

## 7. Efficiency Estimation

<a id="7-efficiency-estimation"></a>

Now we estimate firm-level technical efficiency using two estimators:

1. **JLMS** (Jondrow et al., 1982): Estimates $E[u_i | \varepsilon_i]$, then computes $\widehat{TE}_i = \exp(-\hat{u}_i)$
2. **BC** (Battese & Coelli, 1988): Directly estimates $E[\exp(-u_i) | \varepsilon_i]$

The BC estimator is generally preferred because it avoids the bias introduced by exponentiating a conditional mean (Jensen's inequality).

In [ ]:
# Calculate efficiency using both JLMS and BC estimators
eff_jlms = result.efficiency(estimator="jlms", ci_level=0.95)
eff_bc = result.efficiency(estimator="bc", ci_level=0.95)

# Add to dataset
data["te_jlms"] = eff_jlms["efficiency"].values
data["te_bc"] = eff_bc["efficiency"].values

# Also store inefficiency and confidence intervals from BC
data["ineff_bc"] = eff_bc["inefficiency"].values
data["te_bc_lower"] = eff_bc["ci_lower"].values
data["te_bc_upper"] = eff_bc["ci_upper"].values

# Summary statistics
summary = pd.DataFrame({"JLMS": data["te_jlms"].describe(), "BC": data["te_bc"].describe()})

print("=" * 50)
print("Technical Efficiency Summary")
print("=" * 50)
display(summary)

print(f"\nCorrelation between JLMS and BC: {data['te_jlms'].corr(data['te_bc']):.4f}")
print(f"\nMean difference (JLMS - BC):     {(data['te_jlms'] - data['te_bc']).mean():.4f}")
print(f"Max difference  (JLMS - BC):     {(data['te_jlms'] - data['te_bc']).abs().max():.4f}")

In [ ]:
# JLMS vs BC efficiency comparison scatter plot
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    data["te_jlms"],
    data["te_bc"],
    alpha=0.5,
    s=30,
    color="steelblue",
    edgecolors="navy",
    linewidth=0.3,
)

# 45-degree line
lims = [
    min(data["te_jlms"].min(), data["te_bc"].min()) - 0.01,
    max(data["te_jlms"].max(), data["te_bc"].max()) + 0.01,
]
ax.plot(lims, lims, "r--", linewidth=2, label="45-degree line", alpha=0.7)

ax.set_xlabel("JLMS Technical Efficiency", fontsize=13)
ax.set_ylabel("BC Technical Efficiency", fontsize=13)
ax.set_title("Comparison of JLMS vs BC Efficiency Estimators", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")

# Add correlation annotation
corr = data["te_jlms"].corr(data["te_bc"])
ax.annotate(
    f"Correlation = {corr:.4f}",
    xy=(0.05, 0.95),
    xycoords="axes fraction",
    fontsize=12,
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
    verticalalignment="top",
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "efficiency_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

### Interpretation: JLMS vs BC Efficiency

Key observations from the comparison:

- **High correlation**: The two estimators are highly correlated, meaning the *ranking* of firms is similar regardless of which estimator you use.
- **BC tends to be lower**: The BC estimator typically produces slightly lower efficiency scores than JLMS. This is because $E[\exp(-u)] \leq \exp(-E[u])$ by Jensen's inequality.
- **The difference is small**: For most practical purposes, the two estimators give similar results. However, the BC estimator is theoretically preferred.

**Recommendation**: Use the **BC estimator** for reporting efficiency scores, and JLMS for comparison. Always report which estimator was used.

## 8. Efficiency Visualization

<a id="8-efficiency-visualization"></a>

Visualizing the distribution of efficiency scores helps us understand the overall efficiency landscape in our sample. We examine:
- The overall distribution (histogram + KDE)
- Top and bottom performers
- Efficiency by hospital characteristics (teaching status, location)

In [ ]:
# Try using the built-in plot_efficiency method first
try:
    fig = result.plot_efficiency(
        kind="histogram",
        backend="matplotlib",
        estimator="bc",
        bins=30,
        show_kde=True,
        show_stats=True,
    )
    plt.savefig(FIGURES_DIR / "efficiency_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Built-in plot_efficiency not available ({e}), creating manually...")

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.hist(
        data["te_bc"],
        bins=30,
        density=True,
        alpha=0.7,
        color="steelblue",
        edgecolor="black",
        linewidth=0.5,
        label="BC Efficiency",
    )

    # Add KDE
    kde = stats.gaussian_kde(data["te_bc"])
    x_range = np.linspace(data["te_bc"].min(), data["te_bc"].max(), 200)
    ax.plot(x_range, kde(x_range), "r-", linewidth=2, label="KDE")

    # Add reference lines
    mean_te = data["te_bc"].mean()
    median_te = data["te_bc"].median()
    ax.axvline(mean_te, color="green", linestyle="--", linewidth=2, label=f"Mean: {mean_te:.3f}")
    ax.axvline(
        median_te, color="orange", linestyle=":", linewidth=2, label=f"Median: {median_te:.3f}"
    )

    ax.set_xlabel("Technical Efficiency (BC)", fontsize=13)
    ax.set_ylabel("Density", fontsize=13)
    ax.set_title("Distribution of Technical Efficiency", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)

    # Add statistics box
    stats_text = (
        f"N = {len(data)}\n"
        f"Mean = {mean_te:.3f}\n"
        f"Std = {data['te_bc'].std():.3f}\n"
        f"Min = {data['te_bc'].min():.3f}\n"
        f"Max = {data['te_bc'].max():.3f}"
    )
    ax.text(
        0.02,
        0.75,
        stats_text,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "lightyellow", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "efficiency_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Top 10 most efficient hospitals
print("=" * 70)
print("TOP 10 MOST EFFICIENT HOSPITALS")
print("=" * 70)
top10 = data.nlargest(10, "te_bc")[
    ["hospital_id", "te_bc", "te_bc_lower", "te_bc_upper", "teaching", "urban"]
].reset_index(drop=True)
top10.index = top10.index + 1
top10.columns = ["Hospital ID", "BC Efficiency", "CI Lower", "CI Upper", "Teaching", "Urban"]
display(top10)

print(f"\n{'=' * 70}")
print("BOTTOM 10 LEAST EFFICIENT HOSPITALS")
print("=" * 70)
bottom10 = data.nsmallest(10, "te_bc")[
    ["hospital_id", "te_bc", "te_bc_lower", "te_bc_upper", "teaching", "urban"]
].reset_index(drop=True)
bottom10.index = bottom10.index + 1
bottom10.columns = ["Hospital ID", "BC Efficiency", "CI Lower", "CI Upper", "Teaching", "Urban"]
display(bottom10)

print(
    f"\nEfficiency gap: {data['te_bc'].max():.3f} - {data['te_bc'].min():.3f} = {data['te_bc'].max() - data['te_bc'].min():.3f}"
)
print(
    f"This means the most efficient hospital produces {(data['te_bc'].max() / data['te_bc'].min() - 1) * 100:.1f}% more output"
)
print("than the least efficient hospital, given the same inputs and noise realization.")

In [ ]:
# Efficiency by hospital characteristics
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# By teaching status
teaching_labels = {0: "Non-Teaching", 1: "Teaching"}
data["teaching_label"] = data["teaching"].map(teaching_labels)
sns.boxplot(
    x="teaching_label", y="te_bc", data=data, ax=axes[0], palette=["#FF9800", "#2196F3"], width=0.5
)
axes[0].set_xlabel("Hospital Type", fontsize=12)
axes[0].set_ylabel("Technical Efficiency (BC)", fontsize=12)
axes[0].set_title("Efficiency by Teaching Status", fontsize=13, fontweight="bold")

# Mann-Whitney U test for teaching
teaching_0 = data[data["teaching"] == 0]["te_bc"]
teaching_1 = data[data["teaching"] == 1]["te_bc"]
mw_stat, mw_pval = stats.mannwhitneyu(teaching_0, teaching_1, alternative="two-sided")
axes[0].annotate(
    f"Mann-Whitney U test\nU = {mw_stat:.1f}, p = {mw_pval:.4f}",
    xy=(0.5, 0.02),
    xycoords="axes fraction",
    ha="center",
    fontsize=10,
    bbox={"boxstyle": "round", "facecolor": "lightyellow", "alpha": 0.8},
)

# By urban status
urban_labels = {0: "Rural", 1: "Urban"}
data["urban_label"] = data["urban"].map(urban_labels)
sns.boxplot(
    x="urban_label", y="te_bc", data=data, ax=axes[1], palette=["#4CAF50", "#9C27B0"], width=0.5
)
axes[1].set_xlabel("Location", fontsize=12)
axes[1].set_ylabel("Technical Efficiency (BC)", fontsize=12)
axes[1].set_title("Efficiency by Location", fontsize=13, fontweight="bold")

# Mann-Whitney U test for urban
urban_0 = data[data["urban"] == 0]["te_bc"]
urban_1 = data[data["urban"] == 1]["te_bc"]
mw_stat2, mw_pval2 = stats.mannwhitneyu(urban_0, urban_1, alternative="two-sided")
axes[1].annotate(
    f"Mann-Whitney U test\nU = {mw_stat2:.1f}, p = {mw_pval2:.4f}",
    xy=(0.5, 0.02),
    xycoords="axes fraction",
    ha="center",
    fontsize=10,
    bbox={"boxstyle": "round", "facecolor": "lightyellow", "alpha": 0.8},
)

plt.suptitle("Technical Efficiency by Hospital Characteristics", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "efficiency_by_groups.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary by groups
print("Efficiency by Teaching Status:")
display(data.groupby("teaching_label")["te_bc"].describe().round(4))
print("\nEfficiency by Location:")
display(data.groupby("urban_label")["te_bc"].describe().round(4))

### Interpretation of Efficiency Patterns

The boxplots and statistical tests reveal interesting patterns:

- **Teaching vs. Non-Teaching**: Teaching hospitals may show different efficiency levels due to their dual mission (patient care + education). The Mann-Whitney U test tells us whether this difference is statistically significant.
- **Urban vs. Rural**: Urban hospitals may benefit from economies of scale, better infrastructure, or more competition. Alternatively, they may face higher costs and more complex cases.

**Important caveat**: These are *unconditional* comparisons. The teaching and urban variables are already included in the frontier model, so they affect the *location* of the frontier. The efficiency scores reflect deviations from the group-specific frontier. A more rigorous analysis would use an inefficiency effects model (covered in a later tutorial).

## 9. Frontier Visualization

<a id="9-frontier-visualization"></a>

Visualizing the frontier helps us understand the technology and see how firms relate to the best-practice boundary. We plot the frontier in 2D by showing the relationship between output and one input at a time, holding other inputs at their mean values.

In [ ]:
# Try the built-in plot_frontier method
try:
    fig = result.plot_frontier(
        input_var="log_doctors", kind="2d", backend="matplotlib", show_distance=True
    )
    plt.savefig(FIGURES_DIR / "frontier_2d.png", dpi=300, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Built-in plot_frontier not available ({e}), creating manually...")

    fig, ax = plt.subplots(figsize=(10, 7))

    # Get frontier parameters
    params = result.params
    input_var = "log_doctors"

    # Create range of input values
    x_range = np.linspace(data[input_var].min() - 0.2, data[input_var].max() + 0.2, 200)

    # Calculate frontier prediction (holding other inputs at mean)
    other_vars = ["log_nurses", "log_beds", "teaching", "urban"]
    frontier_pred = params.get("const", params.get("_cons", params.iloc[0]))

    # Add contribution of the focal input
    if input_var in params.index:
        frontier_pred = frontier_pred + params[input_var] * x_range

    # Add contribution of other variables at their means
    for var in other_vars:
        if var in params.index:
            frontier_pred = frontier_pred + params[var] * data[var].mean()

    # Plot frontier
    ax.plot(x_range, frontier_pred, "r-", linewidth=2.5, label="Stochastic Frontier", zorder=3)

    # Plot observations colored by efficiency
    scatter = ax.scatter(
        data[input_var],
        data["log_cases"],
        c=data["te_bc"],
        cmap="RdYlGn",
        s=40,
        alpha=0.7,
        edgecolors="gray",
        linewidth=0.3,
        zorder=2,
    )
    plt.colorbar(scatter, ax=ax, label="Technical Efficiency (BC)", shrink=0.8)

    ax.set_xlabel(f"{input_var}", fontsize=13)
    ax.set_ylabel("log_cases (Output)", fontsize=13)
    ax.set_title(
        "Stochastic Production Frontier\n(other inputs held at mean)",
        fontsize=14,
        fontweight="bold",
    )
    ax.legend(fontsize=12, loc="upper left")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "frontier_2d.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Partial effect plots for each input
input_vars = ["log_doctors", "log_nurses", "log_beds"]
input_labels = ["Log Doctors", "Log Nurses", "Log Beds"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

params = result.params

for ax, var, label in zip(axes, input_vars, input_labels):
    x_range = np.linspace(data[var].min() - 0.1, data[var].max() + 0.1, 200)

    # Calculate frontier: intercept + focal var + other vars at mean
    other_vars = [
        v for v in ["log_doctors", "log_nurses", "log_beds", "teaching", "urban"] if v != var
    ]

    # Start with intercept
    frontier_pred = params.get("const", params.get("_cons", params.iloc[0]))

    # Add focal input contribution
    if var in params.index:
        frontier_pred = frontier_pred + params[var] * x_range

    # Add other variables at their means
    for other_var in other_vars:
        if other_var in params.index:
            frontier_pred = frontier_pred + params[other_var] * data[other_var].mean()

    # Plot frontier
    ax.plot(x_range, frontier_pred, "r-", linewidth=2.5, label="Frontier")

    # Plot observations
    ax.scatter(
        data[var],
        data["log_cases"],
        alpha=0.4,
        s=20,
        color="steelblue",
        edgecolors="navy",
        linewidth=0.2,
    )

    # Add elasticity annotation
    if var in params.index:
        elasticity = params[var]
        ax.annotate(
            f"Elasticity: {elasticity:.3f}",
            xy=(0.05, 0.95),
            xycoords="axes fraction",
            fontsize=11,
            bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
            verticalalignment="top",
        )

    ax.set_xlabel(label, fontsize=12)
    ax.set_ylabel("Log Cases (Output)", fontsize=12)
    ax.set_title(f"Frontier: Output vs {label}", fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)

plt.suptitle(
    "Partial Frontier Plots (other inputs at mean)", fontsize=15, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "frontier_partial_effects.png", dpi=300, bbox_inches="tight")
plt.show()

# Returns to scale
input_elasticities = {var: params.get(var, 0) for var in input_vars if var in params.index}
rts = sum(input_elasticities.values())
print("\nReturns to Scale (sum of input elasticities):")
for var, elast in input_elasticities.items():
    print(f"  {var}: {elast:.4f}")
print(f"  {'=' * 30}")
print(f"  Sum (RTS): {rts:.4f}")
if rts > 1.05:
    print("  Interpretation: Increasing returns to scale")
elif rts < 0.95:
    print("  Interpretation: Decreasing returns to scale")
else:
    print("  Interpretation: Approximately constant returns to scale")

## 10. Comparing Distributional Assumptions

<a id="10-comparing-distributional-assumptions"></a>

The choice of distribution for the inefficiency term $u_i$ affects the estimated efficiency scores. We compare three common specifications:

| Distribution | Key Property | When to Use |
|-------------|-------------|-------------|
| **Half-normal** | Mode at zero | When most firms are expected to be near-efficient |
| **Exponential** | Mode at zero, thinner tail | Similar to half-normal, often gives higher mean efficiency |
| **Truncated normal** | Mode can be away from zero | When there may be a "baseline" level of inefficiency |

We compare these models using:
- **Log-likelihood**: Higher is better (better fit)
- **AIC/BIC**: Lower is better (penalizes complexity)
- **Mean efficiency**: How different are the average efficiency scores?

In [ ]:
# Compare different distributional assumptions
distributions = ["half_normal", "exponential", "truncated_normal"]
dist_labels = ["Half-Normal", "Exponential", "Truncated Normal"]

results_dict = {}
comparison_data = []

for dist, label in zip(distributions, dist_labels):
    print(f"\nEstimating {label} model...")

    try:
        model = StochasticFrontier(
            data=data,
            depvar="log_cases",
            exog=["log_doctors", "log_nurses", "log_beds", "teaching", "urban"],
            frontier="production",
            dist=dist,
        )
        res = model.fit(method="mle", optimizer="L-BFGS-B")

        # Get BC efficiency
        eff = res.efficiency(estimator="bc", ci_level=0.95)
        mean_te = eff["efficiency"].mean()

        results_dict[dist] = {"result": res, "efficiency": eff["efficiency"].values, "label": label}

        comparison_data.append(
            {
                "Distribution": label,
                "Log-Likelihood": res.loglik,
                "AIC": res.aic,
                "BIC": res.bic,
                "N Params": res.nparams,
                "sigma_u": res.sigma_u,
                "sigma_v": res.sigma_v,
                "lambda": res.lambda_param,
                "gamma": res.gamma,
                "Mean TE": mean_te,
                "Std TE": eff["efficiency"].std(),
                "Min TE": eff["efficiency"].min(),
                "Max TE": eff["efficiency"].max(),
                "Converged": res.converged,
            }
        )
        print(f"  Log-lik: {res.loglik:.4f}, AIC: {res.aic:.4f}, Mean TE: {mean_te:.4f}")

    except Exception as e:
        print(f"  ERROR: {e}")

# Create comparison table
comparison_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
display(comparison_df.set_index("Distribution").T)

# Identify best model by AIC and BIC
best_aic = comparison_df.loc[comparison_df["AIC"].idxmin(), "Distribution"]
best_bic = comparison_df.loc[comparison_df["BIC"].idxmin(), "Distribution"]
print(f"\nBest model by AIC: {best_aic}")
print(f"Best model by BIC: {best_bic}")

In [ ]:
# Plot efficiency distributions side by side
n_dists = len(results_dict)
fig, axes = plt.subplots(1, n_dists, figsize=(6 * n_dists, 5))

if n_dists == 1:
    axes = [axes]

colors = ["#2196F3", "#4CAF50", "#FF9800"]

for idx, (dist, info) in enumerate(results_dict.items()):
    ax = axes[idx]
    eff_vals = info["efficiency"]
    label = info["label"]
    color = colors[idx % len(colors)]

    ax.hist(
        eff_vals, bins=30, density=True, alpha=0.7, color=color, edgecolor="black", linewidth=0.5
    )

    # KDE
    kde = stats.gaussian_kde(eff_vals)
    x_range = np.linspace(eff_vals.min(), eff_vals.max(), 200)
    ax.plot(x_range, kde(x_range), "k-", linewidth=2)

    # Mean line
    ax.axvline(
        eff_vals.mean(),
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Mean: {eff_vals.mean():.3f}",
    )

    ax.set_xlabel("Technical Efficiency (BC)", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(f"{label}", fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)

    # Stats box
    stats_text = (
        f"Mean: {eff_vals.mean():.3f}\n"
        f"Std:  {eff_vals.std():.3f}\n"
        f"Min:  {eff_vals.min():.3f}\n"
        f"Max:  {eff_vals.max():.3f}"
    )
    ax.text(
        0.02,
        0.95,
        stats_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "lightyellow", "alpha": 0.8},
    )

plt.suptitle(
    "Efficiency Distributions Under Different Assumptions", fontsize=15, fontweight="bold", y=1.03
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribution_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

# Correlation matrix across distributions
if len(results_dict) > 1:
    eff_compare = pd.DataFrame(
        {info["label"]: info["efficiency"] for info in results_dict.values()}
    )
    print("\nCorrelation of Efficiency Scores Across Distributions:")
    display(eff_compare.corr().round(4))

### Discussion: Choosing the Right Distribution

**Information Criteria (AIC/BIC)**:
- AIC and BIC penalize model complexity. Since the truncated normal has one more parameter ($\mu$), it needs a substantially better log-likelihood to win.
- If AIC and BIC disagree, prefer BIC for consistency (it penalizes complexity more).

**Sensitivity of Rankings**:
- High correlation across distributions suggests the **ranking** of firms is robust, even if the **levels** differ.
- If rankings change dramatically, results are fragile and should be interpreted with caution.

**Practical Guidance**:
1. Start with **half-normal** (simplest, most common)
2. Try **exponential** as a robustness check
3. Use **truncated normal** if there's a theoretical reason to expect a non-zero mode for inefficiency
4. Report results from multiple specifications and note whether conclusions are robust

**Key insight**: The choice of distribution usually matters more for the *level* of efficiency scores than for the *ranking* of firms.

## 11. Exercises

<a id="11-exercises"></a>

These exercises will help you practice the concepts covered in this tutorial. Try to complete them on your own before checking the solutions.

### Exercise 1: Agricultural Efficiency Analysis (Easy)

Load the `farm_data.csv` dataset and:

1. Estimate a Cobb-Douglas production frontier with `log_output` as the dependent variable and `log_land`, `log_labor`, `log_capital` as inputs, with `irrigation` as a control variable
2. Test for the presence of inefficiency
3. Calculate BC efficiency scores
4. Identify the 10 most and least efficient farms
5. Analyze efficiency by farm size category

**Hints**:
- All regressors go in the `exog` parameter (inputs + controls)
- Use `frontier='production'` and `dist='half_normal'`
- For the inefficiency test, you'll need an OLS model first

In [ ]:
# Exercise 1: Your code here
# farm_data = pd.read_csv(DATA_DIR / 'farm_data.csv')
# display(farm_data.head())

# Step 1: Estimate OLS baseline
# X_farm = sm.add_constant(farm_data[['log_land', 'log_labor', 'log_capital', 'irrigation']])
# ols_farm = sm.OLS(farm_data['log_output'], X_farm).fit()

# Step 2: Estimate SFA model
# sfa_farm = StochasticFrontier(
#     data=farm_data,
#     depvar='log_output',
#     exog=['log_land', 'log_labor', 'log_capital', 'irrigation'],
#     frontier='production',
#     dist='half_normal'
# )
# result_farm = sfa_farm.fit(method='mle', optimizer='L-BFGS-B')
# print(result_farm.summary())

# Step 3: Test for inefficiency
# test_farm = inefficiency_presence_test(
#     loglik_sfa=result_farm.loglik,
#     loglik_ols=ols_farm.llf,
#     residuals_ols=ols_farm.resid.values,
#     frontier_type='production',
#     distribution='half_normal'
# )

# Step 4: Calculate efficiency
# eff_farm = result_farm.efficiency(estimator='bc', ci_level=0.95)
# farm_data['te_bc'] = eff_farm['efficiency'].values

# Step 5: Analyze by size category
# display(farm_data.groupby('size_category')['te_bc'].describe())

### Exercise 2: Sensitivity Analysis (Medium)

Using the hospital dataset, conduct a thorough sensitivity analysis:

1. Estimate the SFA model with **only the three input variables** (log_doctors, log_nurses, log_beds) -- no teaching or urban controls
2. Compare the estimated efficiency scores with the full model (which includes teaching and urban)
3. Create a scatter plot of efficiency scores: restricted model vs. full model
4. Compute the rank correlation (Spearman's rho) between the two sets of efficiency scores
5. Discuss: How sensitive are the results to the inclusion of environmental variables?

**Hints**:
- Use `scipy.stats.spearmanr` for rank correlation
- Think about what it means if the efficiency ranking changes substantially

In [ ]:
# Exercise 2: Your code here
# sfa_restricted = StochasticFrontier(
#     data=data,
#     depvar='log_cases',
#     exog=['log_doctors', 'log_nurses', 'log_beds'],
#     frontier='production',
#     dist='half_normal'
# )
# result_restricted = sfa_restricted.fit(method='mle', optimizer='L-BFGS-B')
# eff_restricted = result_restricted.efficiency(estimator='bc', ci_level=0.95)

# # Compare with full model
# from scipy.stats import spearmanr
# rho, p_value = spearmanr(eff_bc['efficiency'], eff_restricted['efficiency'])
# print(f"Spearman rank correlation: {rho:.4f} (p = {p_value:.4f})")

### Exercise 3: Returns to Scale (Advanced)

Test for constant returns to scale in the hospital production frontier:

1. Use the `returns_to_scale_test()` method on the result object
2. Interpret the results: Does the hospital sector exhibit constant, increasing, or decreasing returns to scale?
3. What are the policy implications of your finding?

**Hints**:
- CRS means $\sum \beta_k = 1$ (a 1% increase in all inputs leads to exactly 1% increase in output)
- IRS ($\sum \beta_k > 1$) suggests merging hospitals could be efficient
- DRS ($\sum \beta_k < 1$) suggests splitting large hospitals could be efficient

In [ ]:
# Exercise 3: Your code here
# rts = result.returns_to_scale_test(
#     input_vars=['log_doctors', 'log_nurses', 'log_beds']
# )
# print(f"Returns to Scale: {rts['rts']:.4f}")
# print(f"Standard Error:   {rts['rts_se']:.4f}")
# print(f"Test Statistic:   {rts['test_statistic']:.4f}")
# print(f"P-value:          {rts['pvalue']:.4f}")
# print(f"95% CI:           [{rts['ci'][0]:.4f}, {rts['ci'][1]:.4f}]")
# print(f"Conclusion:       {rts['conclusion']}")

In [ ]:
# ============================================================
# Export results to various formats
# ============================================================

# 1. Export SFA results to LaTeX and HTML
try:
    latex_output = result.to_latex(
        caption="Stochastic Frontier Analysis Results (Half-Normal)", label="tab:sfa_results"
    )
    with open(TABLES_DIR_LATEX / "sfa_results.tex", "w") as f:
        f.write(latex_output)
    print("LaTeX table saved to:", TABLES_DIR_LATEX / "sfa_results.tex")
except Exception as e:
    print(f"LaTeX export: {e}")

try:
    html_output = result.to_html()
    with open(TABLES_DIR_HTML / "sfa_results.html", "w") as f:
        f.write(html_output)
    print("HTML table saved to:", TABLES_DIR_HTML / "sfa_results.html")
except Exception as e:
    print(f"HTML export: {e}")

# 2. Export comparison table
comparison_df.to_csv(BASE_DIR / "outputs" / "distribution_comparison.csv", index=False)

comparison_latex = comparison_df.to_latex(
    index=False,
    float_format="%.4f",
    caption="Comparison of Distributional Assumptions",
    label="tab:dist_comparison",
)
with open(TABLES_DIR_LATEX / "distribution_comparison.tex", "w") as f:
    f.write(comparison_latex)

comparison_html = comparison_df.to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "distribution_comparison.html", "w") as f:
    f.write(comparison_html)

print("Comparison tables saved.")

# 3. Export efficiency summary
eff_summary = data[["hospital_id", "te_jlms", "te_bc", "te_bc_lower", "te_bc_upper"]].copy()
eff_summary.to_csv(BASE_DIR / "outputs" / "efficiency_scores.csv", index=False)
print(f"Efficiency scores saved ({len(eff_summary)} hospitals).")

print("\nAll exports complete!")

## 12. Summary

<a id="12-summary"></a>

In this tutorial, we covered the fundamentals of Stochastic Frontier Analysis:

### Key Concepts
- **SFA** decomposes the error term into **noise** ($v$) and **inefficiency** ($u$), enabling us to measure how far each firm operates from the best-practice frontier.
- The **composed error** $\varepsilon = v - u$ is the foundation of SFA. The one-sided inefficiency term creates negative skewness in production frontiers.
- **Technical efficiency** $TE = \exp(-u) \in (0, 1]$ measures the ratio of observed output to frontier output.

### Main Findings (Hospital Data)
- The SFA model detected statistically significant inefficiency in the hospital sector.
- The variance decomposition ($\gamma$, $\lambda$) confirmed that a meaningful share of total variance is attributable to inefficiency rather than noise.
- Efficiency scores were relatively robust across different distributional assumptions (half-normal, exponential, truncated normal).
- Hospital characteristics (teaching status, location) appear to be associated with efficiency differences.

### Practical Implications
1. **For hospital managers**: The efficiency scores identify which hospitals have the most room for improvement and can guide resource allocation.
2. **For policymakers**: The returns to scale analysis informs decisions about hospital consolidation or expansion.
3. **For researchers**: Always test for the presence of inefficiency before interpreting efficiency scores. Report results from multiple distributional assumptions.

### What's Next?
In the next tutorial (**Tutorial 11B: Panel Data SFA**), we will extend the analysis to:
- Panel data models (Battese-Coelli 1992, 1995)
- True fixed effects and true random effects models
- Time-varying efficiency
- Heterogeneity in inefficiency (inefficiency effects models)

### Key Takeaways

1. **SFA separates noise from inefficiency** -- unlike OLS which lumps all deviations together
2. **Technical Efficiency** $TE = \exp(-u) \in (0, 1]$ measures output relative to the frontier
3. **BC estimator** is preferred over JLMS for efficiency measurement (less biased)
4. **Gamma ($\gamma$)** tells us what proportion of variance is due to inefficiency
5. **Distributional choice** matters -- always check robustness across specifications
6. **Always test** for the presence of inefficiency before interpreting efficiency scores

### Common Pitfalls

Avoid these mistakes when conducting SFA:

1. **Ignoring the skewness check**: If OLS residuals have the "wrong" skewness (positive for production frontiers), the MLE may converge to the OLS solution ($\sigma_u \to 0$). Always check residual skewness before and after estimation.

2. **Confusing frontier type**: Use `frontier='production'` when higher output is better (inefficiency *reduces* output, so $\varepsilon = v - u$). Use `frontier='cost'` when lower cost is better (inefficiency *increases* cost, so $\varepsilon = v + u$). Mixing these up reverses the efficiency scores.

3. **Not testing for inefficiency**: Always run the likelihood ratio test before interpreting efficiency scores. If the test fails to reject $H_0$, the SFA model may not be appropriate and OLS suffices.

4. **Over-interpreting small efficiency differences**: A hospital with $TE = 0.88$ is not necessarily "worse" than one with $TE = 0.90$. Always check confidence intervals and statistical significance.

5. **Forgetting functional form matters**: The Cobb-Douglas specification assumes constant elasticities and unitary elasticity of substitution. Translog is more flexible but has more parameters.

6. **Neglecting multicollinearity**: Log-inputs are often highly correlated. This inflates standard errors and makes individual elasticities unreliable, even if the overall model fits well.

7. **Not checking convergence**: MLE estimation can fail to converge, especially with small samples or when $\sigma_u$ is close to zero. Always check `result.converged` and try different starting values or optimizers if needed.

## 13. References

<a id="13-references"></a>

### Seminal Papers

1. **Aigner, D., Lovell, C.A.K., & Schmidt, P.** (1977). "Formulation and estimation of stochastic frontier production function models." *Journal of Econometrics*, 6(1), 21-37. -- *The foundational paper that introduced the stochastic frontier model.*

2. **Meeusen, W., & van den Broeck, J.** (1977). "Efficiency estimation from Cobb-Douglas production functions with composed error." *International Economic Review*, 18(2), 435-444. -- *Independent discovery of the SFA model, published the same year as Aigner et al.*

3. **Jondrow, J., Lovell, C.A.K., Materov, I.S., & Schmidt, P.** (1982). "On the estimation of technical inefficiency in the stochastic frontier production function model." *Journal of Econometrics*, 19(2-3), 233-238. -- *Introduced the JLMS estimator for firm-level efficiency.*

4. **Battese, G.E., & Coelli, T.J.** (1988). "Prediction of firm-level technical efficiencies with a generalized frontier production function and panel data." *Journal of Econometrics*, 38(3), 387-399. -- *Introduced the BC estimator and panel data extensions.*

### Textbooks

5. **Kumbhakar, S.C., & Lovell, C.A.K.** (2000). *Stochastic Frontier Analysis*. Cambridge University Press. -- *The definitive textbook on SFA methodology.*

6. **Coelli, T.J., Rao, D.S.P., O'Donnell, C.J., & Battese, G.E.** (2005). *An Introduction to Efficiency and Productivity Analysis*. Springer, 2nd edition. -- *Accessible introduction covering both SFA and DEA.*

---

*This tutorial is part of the PanelBox Stochastic Frontier Analysis module. For questions or suggestions, please open an issue on the project repository.*